# PAATRA Step 1: Parameter Allocation Audit

Load small language models and break down where every parameter lives:  
**embeddings (vocabulary)** vs **transformer blocks (reasoning)**.

This validates the core PAATRA observation: small models inherited from big families waste a huge fraction of their parameters on oversized vocabulary tables.

In [ ]:
!pip install -q torch transformers accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM
import gc

def audit_model(model_id: str):
    """Load a model and break down parameter allocation."""
    print(f"\n{'='*60}")
    print(f"  {model_id}")
    print(f"{'='*60}")

    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        device_map="cpu",
    )
    config = model.config

    vocab_size = config.vocab_size
    hidden_dim = config.hidden_size
    num_layers = config.num_hidden_layers
    print(f"\nConfig: vocab={vocab_size:,}  hidden={hidden_dim}  layers={num_layers}")

    embedding_params = 0
    transformer_params = 0
    lm_head_params = 0
    other_params = 0

    for name, param in model.named_parameters():
        count = param.numel()
        if "embed_tokens" in name or "wte" in name:
            embedding_params += count
        elif "lm_head" in name:
            lm_head_params += count
        elif any(k in name for k in ["layers.", "block.", "h."]):
            transformer_params += count
        else:
            other_params += count

    tied = False
    if hasattr(model, "lm_head") and hasattr(model, "get_input_embeddings"):
        emb_weight = model.get_input_embeddings().weight
        head_weight = model.lm_head.weight
        if emb_weight.data_ptr() == head_weight.data_ptr():
            tied = True

    total = sum(p.numel() for p in model.parameters())
    vocab_total = embedding_params
    if not tied:
        vocab_total += lm_head_params

    print(f"\n--- Parameter Breakdown ---")
    print(f"  Input embeddings:   {embedding_params:>12,}  ({embedding_params/total*100:5.1f}%)")
    if tied:
        print(f"  LM head (output):   {'[tied to input]':>12}")
    else:
        print(f"  LM head (output):   {lm_head_params:>12,}  ({lm_head_params/total*100:5.1f}%)")
    print(f"  Transformer blocks: {transformer_params:>12,}  ({transformer_params/total*100:5.1f}%)")
    print(f"  Other (norms etc):  {other_params:>12,}  ({other_params/total*100:5.1f}%)")
    print(f"  {'─'*42}")
    print(f"  TOTAL:              {total:>12,}")

    print(f"\n--- The PAATRA Question ---")
    print(f"  Vocabulary cost:    {vocab_total:,} params = {vocab_total/total*100:.1f}% of model")
    print(f"  Reasoning capacity: {transformer_params:,} params = {transformer_params/total*100:.1f}% of model")
    ratio = vocab_total / transformer_params if transformer_params > 0 else float('inf')
    print(f"  Vocab-to-Transformer ratio: {ratio:.2f}x")

    print(f"\n--- What-If: Reallocate to 10K vocab ---")
    small_vocab_params = 10_000 * hidden_dim
    freed = vocab_total - small_vocab_params
    new_transformer = transformer_params + freed
    print(f"  10K vocab embedding: {small_vocab_params:,} params")
    print(f"  Freed params:        {freed:,}")
    print(f"  New transformer capacity: {new_transformer:,} ({new_transformer/transformer_params:.1f}x current)")

    result = {
        "model": model_id,
        "total": total,
        "vocab": vocab_total,
        "transformer": transformer_params,
        "vocab_pct": vocab_total / total * 100,
        "transformer_pct": transformer_params / total * 100,
        "vocab_size": vocab_size,
        "hidden_dim": hidden_dim,
        "num_layers": num_layers,
        "tied": tied,
    }

    del model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return result

## Config-Only Audit (Gated Models)

These models need HuggingFace auth, so we compute from known architecture specs.

In [ ]:
gated_models = [
    ("Gemma 3 270M", 262_144, 1536, 18, True),
    ("Llama 3.2 1B", 128_256, 2048, 16, False),
    ("Llama 3.2 3B", 128_256, 3072, 28, False),
    ("Gemma 2 2B", 256_000, 2304, 26, True),
]
for name, vocab, hidden, layers, tied in gated_models:
    emb = vocab * hidden
    trans_est = layers * 12 * hidden * hidden
    total_est = emb + trans_est + (0 if tied else emb)
    vocab_cost = emb if tied else emb * 2
    print(name, vocab_cost / total_est * 100)

## Real Model Audit (Ungated Models)

In [ ]:
models_to_audit = [
    "Qwen/Qwen2.5-0.5B",
    "HuggingFaceTB/SmolLM2-360M",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]
results = []
for m in models_to_audit:
    try:
        results.append(audit_model(m))
    except Exception as e:
        print(f"Failed to load {m}: {e}")